# Step 1: Session Reconstruction & Data Integrity
Pipeline Stage: Post-Preprocessing (Step 1)

### Overview
This step runs right after basic data preprocessing. It groups individual event logs into full sessions for behavioral analysis without modifying original data or biasing future models.

In [ ]:
# ============================================================
# STEP 0 — SOURCE FREEZE & INTEGRITY AUDIT
# NO MODIFICATION TO SOURCE FILE
# ============================================================

import pandas as pd
import hashlib
import json
from pathlib import Path

SOURCE = Path("/content/unified_security_events_clean_v1_BACKUP.csv")

print("=" * 70)
print("STEP 0 — SOURCE FREEZE & INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. File existence
# ------------------------------------------------------------
print("\n[1] FILE")
print("Exists:", SOURCE.exists())
print("Path:", SOURCE)

if not SOURCE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE}")

# ------------------------------------------------------------
# 2. SHA-256 of original source
# ------------------------------------------------------------
sha256 = hashlib.sha256()

with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

source_hash = sha256.hexdigest()

print("\n[2] SOURCE SHA-256")
print(source_hash)

# ------------------------------------------------------------
# 3. Load source — READ ONLY
# ------------------------------------------------------------
df = pd.read_csv(SOURCE, low_memory=False)

print("\n[3] SHAPE")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# ------------------------------------------------------------
# 4. Schema fingerprint
# ------------------------------------------------------------
schema_text = "|".join(
    f"{col}:{str(dtype)}"
    for col, dtype in zip(df.columns, df.dtypes)
)

schema_hash = hashlib.sha256(
    schema_text.encode("utf-8")
).hexdigest()

print("\n[4] SCHEMA")
print("Schema SHA-256:", schema_hash)

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col} -> {df[col].dtype}")

# ------------------------------------------------------------
# 5. Duplicate audit
# ------------------------------------------------------------
duplicate_rows = int(df.duplicated().sum())

print("\n[5] DUPLICATES")
print("Duplicate complete rows:", duplicate_rows)

# ------------------------------------------------------------
# 6. Null profile
# ------------------------------------------------------------
null_counts = df.isna().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)

print("\n[6] NULL PROFILE")

if len(null_counts) == 0:
    print("No missing values.")
else:
    print(null_counts.to_string())

# ------------------------------------------------------------
# 7. Required columns
# ------------------------------------------------------------
required_columns = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset",
]

missing_required = [
    c for c in required_columns
    if c not in df.columns
]

extra_columns = [
    c for c in df.columns
    if c not in required_columns
]

print("\n[7] REQUIRED COLUMN CHECK")
print("Missing required columns:", missing_required)
print("Extra columns:", extra_columns)

# ------------------------------------------------------------
# 8. Timestamp integrity
# ------------------------------------------------------------
timestamp = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

invalid_timestamps = int(timestamp.isna().sum())

print("\n[8] TIMESTAMP")
print("Invalid timestamps:", invalid_timestamps)

if invalid_timestamps == 0:
    print("Minimum:", timestamp.min())
    print("Maximum:", timestamp.max())

# ------------------------------------------------------------
# 9. Session profile
# ------------------------------------------------------------
session_null = int(df["session_id"].isna().sum())

valid_session_mask = df["session_id"].notna()
unique_sessions = int(df.loc[valid_session_mask, "session_id"].nunique())

duplicate_session_rows = int(
    df.loc[valid_session_mask, "session_id"].duplicated().sum()
)

events_per_session = (
    df.loc[valid_session_mask]
      .groupby("session_id")
      .size()
)

print("\n[9] SESSION PROFILE")
print("Null session_id:", session_null)
print("Valid session_id rows:", int(valid_session_mask.sum()))
print("Unique sessions:", unique_sessions)

if len(events_per_session):
    print("Min events/session:", int(events_per_session.min()))
    print("Median events/session:", float(events_per_session.median()))
    print("Mean events/session:", float(events_per_session.mean()))
    print("Max events/session:", int(events_per_session.max()))

# ------------------------------------------------------------
# 10. Source distribution
# ------------------------------------------------------------
print("\n[10] SOURCE DATASET DISTRIBUTION")
print(df["source_dataset"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 11. Honeypot distribution
# ------------------------------------------------------------
print("\n[11] HONEYPOT DISTRIBUTION")
print(df["honeypot"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 12. Protocol distribution
# ------------------------------------------------------------
print("\n[12] PROTOCOL DISTRIBUTION")
print(df["protocol"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 13. Integrity snapshot
# ------------------------------------------------------------
snapshot = {
    "source_file": SOURCE.name,
    "source_sha256": source_hash,
    "schema_sha256": schema_hash,
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "duplicate_complete_rows": duplicate_rows,
    "null_session_ids": session_null,
    "valid_session_rows": int(valid_session_mask.sum()),
    "unique_sessions": unique_sessions,
    "invalid_timestamps": invalid_timestamps,
}

snapshot_hash = hashlib.sha256(
    json.dumps(
        snapshot,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("\n[13] INTEGRITY SNAPSHOT")
print(json.dumps(snapshot, indent=2))
print("\nIntegrity Snapshot SHA-256:")
print(snapshot_hash)

# ------------------------------------------------------------
# 14. FINAL STATUS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 0 STATUS")
print("=" * 70)

checks = {
    "file_exists": SOURCE.exists(),
    "required_columns_present": len(missing_required) == 0,
    "no_invalid_timestamps": invalid_timestamps == 0,
    "no_duplicate_complete_rows": duplicate_rows == 0,
}

for name, status in checks.items():
    print(f"{name}: {'PASS' if status else 'FAIL'}")

print("\nSOURCE FILE WAS NOT MODIFIED.")
print("NO FEATURES CREATED.")
print("NO GEOIP ENRICHMENT.")
print("NO SESSION RECONSTRUCTION.")
print("NO ML.")
print("=" * 70)

STEP 0 — SOURCE FREEZE & INTEGRITY AUDIT

[1] FILE
Exists: True
Path: /content/unified_security_events_clean_v1_BACKUP.csv

[2] SOURCE SHA-256
a1639f172eaca4a3c3a4ed02045b33e7d8152c5ec71129e7d33ad8f479edbcca

[3] SHAPE
Rows: 58895
Columns: 24

[4] SCHEMA
Schema SHA-256: 630f3abcf454ef8650d2d55f1110248d662ff630a393f7a1a3b359b5de8dc68a

Columns:
01. timestamp -> object
02. honeypot -> object
03. protocol -> object
04. source_ip -> object
05. source_port -> float64
06. destination_ip -> object
07. destination_port -> float64
08. session_id -> object
09. username -> object
10. password -> object
11. user_attempt -> int64
12. pass_attempt -> int64
13. auth_attempt -> int64
14. anonymous -> int64
15. command_count -> int64
16. country -> object
17. country_code -> object
18. asn -> float64
19. ip_reputation -> object
20. attack_type -> object
21. mitre_technique -> object
22. attack_stage -> object
23. severity -> object
24. source_dataset -> object

[5] DUPLICATES
Duplicate complete rows: 0

In [ ]:
# ============================================================
# STEP 1 — GEOIP / ENRICHMENT INPUT AUDIT
# ============================================================
# READ ONLY
# NO SOURCE MODIFICATION
# NO ML
# NO SESSION RECONSTRUCTION
# NO TARGET CREATION
# ============================================================

import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

SOURCE = Path("/content/unified_security_events_clean_v1_BACKUP.csv")

print("=" * 70)
print("STEP 1 — GEOIP / ENRICHMENT INPUT AUDIT")
print("=" * 70)

df = pd.read_csv(SOURCE, low_memory=False)

# ------------------------------------------------------------
# 1. Source identity
# ------------------------------------------------------------
print("\n[1] SOURCE IDENTITY")
print("Rows:", len(df))
print("Columns:", len(df.columns))

sha256 = hashlib.sha256()
with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("SHA-256:", sha256.hexdigest())

# ------------------------------------------------------------
# 2. Source IP availability
# ------------------------------------------------------------
print("\n[2] SOURCE IP QUALITY")

source_ip_null = int(df["source_ip"].isna().sum())
source_ip_unique = int(df["source_ip"].nunique(dropna=True))

print("Null source_ip:", source_ip_null)
print("Unique source_ip:", source_ip_unique)

print("\nTop source IPs:")
print(df["source_ip"].value_counts(dropna=False).head(20).to_string())

# ------------------------------------------------------------
# 3. Existing GeoIP fields
# ------------------------------------------------------------
geo_columns = [
    "country",
    "country_code",
    "asn",
    "ip_reputation"
]

print("\n[3] EXISTING ENRICHMENT FIELDS")

for col in geo_columns:
    print(f"\n{col}")
    print("  dtype:", df[col].dtype)
    print("  non-null:", int(df[col].notna().sum()))
    print("  null:", int(df[col].isna().sum()))
    print("  unique:", int(df[col].nunique(dropna=True)))

# ------------------------------------------------------------
# 4. GeoIP consistency by source IP
# ------------------------------------------------------------
print("\n[4] GEOIP CONSISTENCY PER SOURCE IP")

for col in ["country", "country_code", "asn", "ip_reputation"]:

    consistency = (
        df.dropna(subset=["source_ip"])
          .groupby("source_ip")[col]
          .nunique(dropna=True)
    )

    problematic = int((consistency > 1).sum())

    print(
        f"{col}: "
        f"{problematic} source IPs have multiple non-null values"
    )

# ------------------------------------------------------------
# 5. Country ↔ country_code consistency
# ------------------------------------------------------------
print("\n[5] COUNTRY / COUNTRY_CODE CONSISTENCY")

country_code_pairs = (
    df[["country", "country_code"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["country", "country_code"])
)

print(country_code_pairs.to_string(index=False))

print("\nUnique country values:")
print(df["country"].value_counts(dropna=False).to_string())

print("\nUnique country_code values:")
print(df["country_code"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 6. ASN quality
# ------------------------------------------------------------
print("\n[6] ASN QUALITY")

print("Non-null ASN:", int(df["asn"].notna().sum()))
print("Null ASN:", int(df["asn"].isna().sum()))
print("Unique ASN:", int(df["asn"].nunique(dropna=True)))

print("\nTop ASN values:")
print(df["asn"].value_counts(dropna=False).head(20).to_string())

# ------------------------------------------------------------
# 7. Reputation quality
# ------------------------------------------------------------
print("\n[7] IP REPUTATION QUALITY")

print(
    df["ip_reputation"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 8. Enrichment coverage by honeypot
# ------------------------------------------------------------
print("\n[8] ENRICHMENT COVERAGE BY HONEYPOT")

coverage = []

for honeypot, group in df.groupby("honeypot", dropna=False):

    row = {
        "honeypot": honeypot,
        "rows": len(group),
        "source_ip_non_null": group["source_ip"].notna().sum(),
        "country_non_null": group["country"].notna().sum(),
        "country_code_non_null": group["country_code"].notna().sum(),
        "asn_non_null": group["asn"].notna().sum(),
        "reputation_non_null": group["ip_reputation"].notna().sum(),
    }

    coverage.append(row)

coverage_df = pd.DataFrame(coverage)

print(coverage_df.to_string(index=False))

# ------------------------------------------------------------
# 9. Enrichment coverage by source dataset
# ------------------------------------------------------------
print("\n[9] ENRICHMENT COVERAGE BY SOURCE DATASET")

coverage_source = []

for source_dataset, group in df.groupby(
    "source_dataset",
    dropna=False
):

    coverage_source.append({
        "source_dataset": source_dataset,
        "rows": len(group),
        "source_ip_non_null": group["source_ip"].notna().sum(),
        "country_non_null": group["country"].notna().sum(),
        "country_code_non_null": group["country_code"].notna().sum(),
        "asn_non_null": group["asn"].notna().sum(),
        "reputation_non_null": group["ip_reputation"].notna().sum(),
    })

coverage_source_df = pd.DataFrame(coverage_source)

print(coverage_source_df.to_string(index=False))

# ------------------------------------------------------------
# 10. Existing enrichment provenance check
# ------------------------------------------------------------
print("\n[10] ENRICHMENT PROVENANCE CHECK")

possible_provenance = [
    c for c in df.columns
    if any(
        token in c.lower()
        for token in [
            "geo",
            "geoip",
            "enrich",
            "reputation_source",
            "asn_source",
            "country_source"
        ]
    )
]

print("Possible provenance columns:")
print(possible_provenance)

# ------------------------------------------------------------
# 11. FINAL DECISION GATE
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 1 AUDIT COMPLETE")
print("=" * 70)

print("""
NO SOURCE FILE WAS MODIFIED.
NO NEW COLUMNS WERE CREATED.
NO VALUES WERE FILLED.
NO GEOIP VALUES WERE REGENERATED.
NO ML FEATURES WERE CREATED.
NO SESSION FEATURES WERE CREATED.
""")

print("Next decision:")
print("""
A) Existing enrichment is trustworthy and traceable
   -> preserve it and continue.

B) Existing enrichment lacks provenance / has inconsistencies
   -> rebuild enrichment into a NEW v2 dataset.

C) Some enrichment fields are reliable and others are not
   -> preserve reliable fields and rebuild only the affected fields.
""")

print("=" * 70)

STEP 1 — GEOIP / ENRICHMENT INPUT AUDIT

[1] SOURCE IDENTITY
Rows: 58895
Columns: 24
SHA-256: a1639f172eaca4a3c3a4ed02045b33e7d8152c5ec71129e7d33ad8f479edbcca

[2] SOURCE IP QUALITY
Null source_ip: 0
Unique source_ip: 1139

Top source IPs:
source_ip
173.231.185.164    4155
47.239.170.26      3743
170.64.153.215     3543
196.251.84.225     2863
170.64.232.83      2607
209.38.85.5        2360
209.38.82.85       2358
134.199.160.187    2357
95.182.115.26      1679
118.31.52.241      1256
142.93.191.172     1254
68.183.90.191      1252
138.68.170.24      1251
34.55.158.124      1251
138.124.30.225     1249
143.198.96.196     1246
103.140.238.42     1246
80.90.185.250      1235
8.138.44.199       1225
209.38.82.168      1186

[3] EXISTING ENRICHMENT FIELDS

country
  dtype: object
  non-null: 55009
  null: 3886
  unique: 4

country_code
  dtype: object
  non-null: 55009
  null: 3886
  unique: 4

asn
  dtype: float64
  non-null: 55009
  null: 3886
  unique: 20

ip_reputation
  dtype: object


In [ ]:
# ============================================================
# STEP 1A.1 — GEOIP ARCHIVE INTEGRITY CHECK
# READ ONLY — NO EXTRACTION
# ============================================================

from pathlib import Path
import hashlib
import gzip

BASE = Path("/content")

archives = [
    BASE / "GeoLite2-City_20260807.tar.gz",
    BASE / "GeoLite2-ASN_20260807 (1).tar.gz",
]

print("=" * 70)
print("GEOIP ARCHIVE INTEGRITY CHECK")
print("=" * 70)

for archive in archives:

    print("\n" + "-" * 70)
    print("FILE:", archive.name)

    if not archive.exists():
        print("STATUS: MISSING")
        continue

    size_mb = archive.stat().st_size / (1024 * 1024)

    print("Exists:", True)
    print(f"Size: {size_mb:.2f} MB")

    # SHA-256
    sha256 = hashlib.sha256()

    try:
        with open(archive, "rb") as f:
            while True:
                chunk = f.read(1024 * 1024)
                if not chunk:
                    break
                sha256.update(chunk)

        print("SHA-256:", sha256.hexdigest())

    except Exception as e:
        print("SHA-256 ERROR:", repr(e))

    # GZIP integrity test
    try:
        with gzip.open(archive, "rb") as f:
            total_read = 0

            while True:
                chunk = f.read(1024 * 1024)

                if not chunk:
                    break

                total_read += len(chunk)

        print("GZIP TEST: PASS")
        print("Decompressed bytes:", total_read)

    except Exception as e:
        print("GZIP TEST: FAIL")
        print("Error:", repr(e))

print("\n" + "=" * 70)
print("ARCHIVE INTEGRITY CHECK COMPLETE")
print("NO ARCHIVE WAS MODIFIED")
print("NO EVENT DATA WAS MODIFIED")
print("=" * 70)

GEOIP ARCHIVE INTEGRITY CHECK

----------------------------------------------------------------------
FILE: GeoLite2-City_20260807.tar.gz
Exists: True
Size: 30.94 MB
SHA-256: 82bc346be68937b447a91d298d2f79839eff0d79af9f0fa9271a05fb20ab620c
GZIP TEST: PASS
Decompressed bytes: 65595904

----------------------------------------------------------------------
FILE: GeoLite2-ASN_20260807 (1).tar.gz
Exists: True
Size: 6.11 MB
SHA-256: eb989843b6e60284e5eefbe53e000344d8c2fff842f3dee338fefe7a26b0898b
GZIP TEST: PASS
Decompressed bytes: 12087296

ARCHIVE INTEGRITY CHECK COMPLETE
NO ARCHIVE WAS MODIFIED
NO EVENT DATA WAS MODIFIED


In [ ]:
# ============================================================
# STEP 1B — SAFE EXTRACTION OF CITY + ASN DATABASES
# READ SOURCE ARCHIVES ONLY
# ============================================================

from pathlib import Path
import tarfile

BASE = Path("/content")

city_archive = BASE / "GeoLite2-City_20260807.tar.gz"
asn_archive  = BASE / "GeoLite2-ASN_20260807 (1).tar.gz"

city_dir = BASE / "geolite_city_v1"
asn_dir  = BASE / "geolite_asn_v1"

for d in [city_dir, asn_dir]:
    if d.exists():
        raise FileExistsError(
            f"Directory already exists: {d}\n"
            "Stop to avoid overwriting an existing extraction."
        )

city_dir.mkdir()
asn_dir.mkdir()

def extract_mmdb(archive_path, output_dir, label):
    print("\n" + "-" * 70)
    print(label)
    print("Archive:", archive_path.name)

    with tarfile.open(archive_path, "r:gz") as tar:
        members = [
            m for m in tar.getmembers()
            if m.isfile() and m.name.endswith(".mmdb")
        ]

        if not members:
            raise RuntimeError(
                f"No .mmdb database found in {archive_path.name}"
            )

        print("MMDB files found:")

        for m in members:
            print(" -", m.name)
            tar.extract(m, path=output_dir)

    print("Extraction: PASS")


extract_mmdb(
    city_archive,
    city_dir,
    "GEOIP CITY DATABASE"
)

extract_mmdb(
    asn_archive,
    asn_dir,
    "GEOIP ASN DATABASE"
)

print("\n" + "=" * 70)
print("STEP 1B COMPLETE")
print("=" * 70)

print("\nCITY DATABASE FILES:")
for p in city_dir.rglob("*.mmdb"):
    print("-", p)

print("\nASN DATABASE FILES:")
for p in asn_dir.rglob("*.mmdb"):
    print("-", p)

print("\nSOURCE EVENTS WAS NOT MODIFIED.")
print("NO ENRICHMENT WAS PERFORMED.")
print("NO ML FEATURES WERE CREATED.")
print("=" * 70)


----------------------------------------------------------------------
GEOIP CITY DATABASE
Archive: GeoLite2-City_20260807.tar.gz
MMDB files found:
 - GeoLite2-City_20260807/GeoLite2-City.mmdb


/tmp/ipykernel_906/3521787082.py:47: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(m, path=output_dir)


Extraction: PASS

----------------------------------------------------------------------
GEOIP ASN DATABASE
Archive: GeoLite2-ASN_20260807 (1).tar.gz
MMDB files found:
 - GeoLite2-ASN_20260807/GeoLite2-ASN.mmdb
Extraction: PASS

STEP 1B COMPLETE

CITY DATABASE FILES:
- /content/geolite_city_v1/GeoLite2-City_20260807/GeoLite2-City.mmdb

ASN DATABASE FILES:
- /content/geolite_asn_v1/GeoLite2-ASN_20260807/GeoLite2-ASN.mmdb

SOURCE EVENTS WAS NOT MODIFIED.
NO ENRICHMENT WAS PERFORMED.
NO ML FEATURES WERE CREATED.


In [ ]:
# Install the reader required to validate MaxMind MMDB databases
!pip -q install maxminddb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.3/101.3 kB 3.2 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STEP 1C — GEOIP MMDB READABILITY & DATABASE VALIDATION
# READ ONLY
# ============================================================

from pathlib import Path

city_db = Path(
    "/content/geolite_city_v1/"
    "GeoLite2-City_20260807/GeoLite2-City.mmdb"
)

asn_db = Path(
    "/content/geolite_asn_v1/"
    "GeoLite2-ASN_20260807/GeoLite2-ASN.mmdb"
)

print("=" * 70)
print("STEP 1C — GEOIP DATABASE VALIDATION")
print("=" * 70)

for label, path in [
    ("CITY", city_db),
    ("ASN", asn_db)
]:
    print("\n" + "-" * 70)
    print(label)
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        print("Size (MB):", round(path.stat().st_size / (1024**2), 2))

        if path.stat().st_size == 0:
            raise RuntimeError(f"{label} database is empty.")

print("\nChecking maxminddb reader...")

try:
    import maxminddb
    print("maxminddb: AVAILABLE")
except ImportError:
    raise ImportError(
        "maxminddb package is not installed. "
        "Install it before continuing."
    )

# Open and inspect metadata
for label, path in [
    ("CITY", city_db),
    ("ASN", asn_db)
]:
    print("\n" + "-" * 70)
    print(f"{label} DATABASE READ TEST")

    with maxminddb.open_database(str(path)) as reader:
        metadata = reader.metadata()

        print("Database type:", metadata.database_type)
        print("Build epoch:", metadata.build_epoch)
        print("Binary format:", metadata.binary_format_major_version,
              metadata.binary_format_minor_version)

print("\n" + "=" * 70)
print("STEP 1C COMPLETE — DATABASES ARE READABLE")
print("=" * 70)

print("NO EVENT DATA MODIFIED")
print("NO ENRICHMENT PERFORMED")
print("NO ML FEATURES CREATED")

STEP 1C — GEOIP DATABASE VALIDATION

----------------------------------------------------------------------
CITY
Path: /content/geolite_city_v1/GeoLite2-City_20260807/GeoLite2-City.mmdb
Exists: True
Size (MB): 62.55

----------------------------------------------------------------------
ASN
Path: /content/geolite_asn_v1/GeoLite2-ASN_20260807/GeoLite2-ASN.mmdb
Exists: True
Size (MB): 11.52

Checking maxminddb reader...
maxminddb: AVAILABLE

----------------------------------------------------------------------
CITY DATABASE READ TEST
Database type: GeoLite2-City
Build epoch: 1786080645
Binary format: 2 0

----------------------------------------------------------------------
ASN DATABASE READ TEST
Database type: GeoLite2-ASN
Build epoch: 1786090543
Binary format: 2 0

STEP 1C COMPLETE — DATABASES ARE READABLE
NO EVENT DATA MODIFIED
NO ENRICHMENT PERFORMED
NO ML FEATURES CREATED


In [ ]:
# ============================================================
# STEP 1D — GEOIP ENRICHMENT V2 — CORRECTED
# CITY + ASN
#
# SOURCE IS READ ONLY
# OUTPUT IS A NEW V2 DATASET
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import maxminddb
import hashlib
import json
from datetime import datetime, timezone

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = Path("/content")

SOURCE = BASE / "unified_security_events_clean_v1_BACKUP.csv"

CITY_DB = (
    BASE
    / "geolite_city_v1/GeoLite2-City_20260807/GeoLite2-City.mmdb"
)

ASN_DB = (
    BASE
    / "geolite_asn_v1/GeoLite2-ASN_20260807/GeoLite2-ASN.mmdb"
)

OUTPUT = BASE / "unified_security_events_geoip_v2.csv"
AUDIT = BASE / "geoip_enrichment_v2_audit.json"

# ------------------------------------------------------------
# SAFETY
# ------------------------------------------------------------

assert SOURCE.exists()
assert CITY_DB.exists()
assert ASN_DB.exists()

if OUTPUT.exists():
    raise FileExistsError(
        f"Output already exists:\n{OUTPUT}\n"
        "Delete/rename it only if you intentionally want a new run."
    )

# ------------------------------------------------------------
# SHA256
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


source_sha_before = sha256_file(SOURCE)
city_sha = sha256_file(CITY_DB)
asn_sha = sha256_file(ASN_DB)

print("=" * 70)
print("STEP 1D — GEOIP ENRICHMENT V2 — CORRECTED")
print("=" * 70)

print("\nSOURCE SHA-256:")
print(source_sha_before)

print("\nCITY DB SHA-256:")
print(city_sha)

print("\nASN DB SHA-256:")
print(asn_sha)

# ------------------------------------------------------------
# LOAD SOURCE
# ------------------------------------------------------------

df = pd.read_csv(
    SOURCE,
    low_memory=False
)

print("\nSOURCE SHAPE:")
print(df.shape)

assert df.shape == (58895, 24)
assert "source_ip" in df.columns

# ------------------------------------------------------------
# PRESERVE ORIGINAL ENRICHMENT
# ------------------------------------------------------------

df["original_country"] = df["country"]
df["original_country_code"] = df["country_code"]
df["original_asn"] = df["asn"]

# ------------------------------------------------------------
# UNIQUE IPs
# ------------------------------------------------------------

unique_ips = (
    df["source_ip"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("\nUnique source IPs:", len(unique_ips))

# ------------------------------------------------------------
# CITY LOOKUP
# ------------------------------------------------------------

city_rows = []

print("\nRunning GeoLite2-City lookups...")

with maxminddb.open_database(str(CITY_DB)) as reader:

    for ip in unique_ips:

        result = reader.get(ip)

        row = {
            "source_ip": ip,
            "geoip_country_v2": None,
            "geoip_country_code_v2": None,
            "geoip_registered_country_v2": None,
            "geoip_city_v2": None,
            "geoip_latitude_v2": None,
            "geoip_longitude_v2": None,
            "geoip_accuracy_radius_v2": None,
        }

        if result is not None:

            country = result.get("country", {})
            registered = result.get(
                "registered_country", {}
            )
            city = result.get("city", {})
            location = result.get("location", {})

            row["geoip_country_v2"] = (
                country.get("names", {}).get("en")
            )

            row["geoip_country_code_v2"] = (
                country.get("iso_code")
            )

            row["geoip_registered_country_v2"] = (
                registered.get("names", {}).get("en")
            )

            row["geoip_city_v2"] = (
                city.get("names", {}).get("en")
            )

            row["geoip_latitude_v2"] = (
                location.get("latitude")
            )

            row["geoip_longitude_v2"] = (
                location.get("longitude")
            )

            row["geoip_accuracy_radius_v2"] = (
                location.get("accuracy_radius")
            )

        city_rows.append(row)

city_lookup = pd.DataFrame(city_rows)

# ------------------------------------------------------------
# ASN LOOKUP
# ------------------------------------------------------------

asn_rows = []

print("Running GeoLite2-ASN lookups...")

with maxminddb.open_database(str(ASN_DB)) as reader:

    for ip in unique_ips:

        result = reader.get(ip)

        row = {
            "source_ip": ip,
            "geoip_asn_v2": None,
            "geoip_asn_org_v2": None,
        }

        if result is not None:

            row["geoip_asn_v2"] = (
                result.get("autonomous_system_number")
            )

            row["geoip_asn_org_v2"] = (
                result.get(
                    "autonomous_system_organization"
                )
            )

        asn_rows.append(row)

asn_lookup = pd.DataFrame(asn_rows)

print("Lookups complete.")

# ------------------------------------------------------------
# LOOKUP TABLE VALIDATION
# ------------------------------------------------------------

assert len(city_lookup) == len(unique_ips)
assert len(asn_lookup) == len(unique_ips)

assert city_lookup["source_ip"].is_unique
assert asn_lookup["source_ip"].is_unique

# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

df = df.merge(
    city_lookup,
    on="source_ip",
    how="left",
    validate="many_to_one"
)

df = df.merge(
    asn_lookup,
    on="source_ip",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# PROVENANCE
# ------------------------------------------------------------

df["geoip_city_database"] = "GeoLite2-City_20260807"
df["geoip_asn_database"] = "GeoLite2-ASN_20260807"
df["geoip_enrichment_version"] = "v2"

# ------------------------------------------------------------
# COVERAGE
# ------------------------------------------------------------

country_coverage = (
    df["geoip_country_v2"].notna().sum()
)

country_code_coverage = (
    df["geoip_country_code_v2"].notna().sum()
)

city_coverage = (
    df["geoip_city_v2"].notna().sum()
)

latitude_coverage = (
    df["geoip_latitude_v2"].notna().sum()
)

longitude_coverage = (
    df["geoip_longitude_v2"].notna().sum()
)

asn_coverage = (
    df["geoip_asn_v2"].notna().sum()
)

asn_org_coverage = (
    df["geoip_asn_org_v2"].notna().sum()
)

# ------------------------------------------------------------
# OLD VS NEW CONSISTENCY
# ------------------------------------------------------------

country_comparable = (
    df["original_country"].notna()
    & df["geoip_country_v2"].notna()
)

country_match = (
    df.loc[
        country_comparable,
        "original_country"
    ]
    == df.loc[
        country_comparable,
        "geoip_country_v2"
    ]
)

country_match_count = int(country_match.sum())
country_comparable_count = int(country_comparable.sum())

country_mismatch_count = (
    country_comparable_count
    - country_match_count
)

country_code_comparable = (
    df["original_country_code"].notna()
    & df["geoip_country_code_v2"].notna()
)

country_code_match = (
    df.loc[
        country_code_comparable,
        "original_country_code"
    ]
    == df.loc[
        country_code_comparable,
        "geoip_country_code_v2"
    ]
)

country_code_match_count = int(
    country_code_match.sum()
)

country_code_comparable_count = int(
    country_code_comparable.sum()
)

country_code_mismatch_count = (
    country_code_comparable_count
    - country_code_match_count
)

# ASN comparison
asn_comparable = (
    df["original_asn"].notna()
    & df["geoip_asn_v2"].notna()
)

asn_match = (
    pd.to_numeric(
        df.loc[asn_comparable, "original_asn"],
        errors="coerce"
    )
    == pd.to_numeric(
        df.loc[asn_comparable, "geoip_asn_v2"],
        errors="coerce"
    )
)

asn_match_count = int(asn_match.sum())
asn_comparable_count = int(asn_comparable.sum())

asn_mismatch_count = (
    asn_comparable_count
    - asn_match_count
)

# ------------------------------------------------------------
# SAVE V2
# ------------------------------------------------------------

df.to_csv(
    OUTPUT,
    index=False
)

output_sha = sha256_file(OUTPUT)

# ------------------------------------------------------------
# SOURCE INTEGRITY AFTER RUN
# ------------------------------------------------------------

source_sha_after = sha256_file(SOURCE)

assert source_sha_before == source_sha_after

# ------------------------------------------------------------
# AUDIT
# ------------------------------------------------------------

audit = {
    "step": "STEP 1D — GEOIP ENRICHMENT V2",

    "source_file": SOURCE.name,
    "source_sha256_before": source_sha_before,
    "source_sha256_after": source_sha_after,

    "output_file": OUTPUT.name,
    "output_sha256": output_sha,

    "source_rows": int(len(pd.read_csv(
        SOURCE,
        low_memory=False
    ))),

    "output_rows": int(len(df)),

    "source_columns": 24,
    "output_columns": int(len(df.columns)),

    "unique_source_ips": int(len(unique_ips)),

    "geoip_country_coverage": int(
        country_coverage
    ),

    "geoip_country_code_coverage": int(
        country_code_coverage
    ),

    "geoip_city_coverage": int(
        city_coverage
    ),

    "geoip_latitude_coverage": int(
        latitude_coverage
    ),

    "geoip_longitude_coverage": int(
        longitude_coverage
    ),

    "geoip_asn_coverage": int(
        asn_coverage
    ),

    "geoip_asn_org_coverage": int(
        asn_org_coverage
    ),

    "country_comparable_rows": country_comparable_count,
    "country_matches": country_match_count,
    "country_mismatches": country_mismatch_count,

    "country_code_comparable_rows":
        country_code_comparable_count,

    "country_code_matches":
        country_code_match_count,

    "country_code_mismatches":
        country_code_mismatch_count,

    "asn_comparable_rows":
        asn_comparable_count,

    "asn_matches":
        asn_match_count,

    "asn_mismatches":
        asn_mismatch_count,

    "city_database":
        "GeoLite2-City_20260807",

    "city_database_sha256":
        city_sha,

    "asn_database":
        "GeoLite2-ASN_20260807",

    "asn_database_sha256":
        asn_sha,

    "enrichment_version": "v2",

    "source_unchanged": True,

    "created_utc":
        datetime.now(timezone.utc).isoformat()
}

with open(AUDIT, "w") as f:
    json.dump(
        audit,
        f,
        indent=2
    )

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GEOIP V2 ENRICHMENT REPORT")
print("=" * 70)

print("\nOutput shape:", df.shape)

print("\nCOVERAGE")
print(
    f"Country       : {country_coverage}/{len(df)} "
    f"({country_coverage/len(df)*100:.2f}%)"
)

print(
    f"Country Code  : {country_code_coverage}/{len(df)} "
    f"({country_code_coverage/len(df)*100:.2f}%)"
)

print(
    f"City          : {city_coverage}/{len(df)} "
    f"({city_coverage/len(df)*100:.2f}%)"
)

print(
    f"Latitude      : {latitude_coverage}/{len(df)} "
    f"({latitude_coverage/len(df)*100:.2f}%)"
)

print(
    f"Longitude     : {longitude_coverage}/{len(df)} "
    f"({longitude_coverage/len(df)*100:.2f}%)"
)

print(
    f"ASN           : {asn_coverage}/{len(df)} "
    f"({asn_coverage/len(df)*100:.2f}%)"
)

print(
    f"ASN Org       : {asn_org_coverage}/{len(df)} "
    f"({asn_org_coverage/len(df)*100:.2f}%)"
)

print("\nOLD vs NEW")

print(
    "Country matches    :",
    country_match_count
)

print(
    "Country mismatches :",
    country_mismatch_count
)

print(
    "Country-code matches    :",
    country_code_match_count
)

print(
    "Country-code mismatches :",
    country_code_mismatch_count
)

print(
    "ASN matches    :",
    asn_match_count
)

print(
    "ASN mismatches :",
    asn_mismatch_count
)

print("\nTOP GEOIP COUNTRIES")
print(
    df["geoip_country_v2"]
    .value_counts(dropna=False)
    .head(10)
)

print("\nTOP GEOIP ASN")
print(
    df["geoip_asn_v2"]
    .value_counts(dropna=False)
    .head(10)
)

print("\nSOURCE INTEGRITY")
print("Before:", source_sha_before)
print("After :", source_sha_after)
print("PASS  :", source_sha_before == source_sha_after)

print("\nOUTPUT:")
print(OUTPUT)

print("\nAUDIT:")
print(AUDIT)

print("\nOUTPUT SHA-256:")
print(output_sha)

print("\n" + "=" * 70)
print("STEP 1D COMPLETE")
print("=" * 70)

print("SOURCE FILE WAS NOT MODIFIED.")
print("GEOIP V2 CREATED.")
print("NO ML FEATURES CREATED.")
print("NO SESSION FEATURES CREATED.")

STEP 1D — GEOIP ENRICHMENT V2 — CORRECTED

SOURCE SHA-256:
a1639f172eaca4a3c3a4ed02045b33e7d8152c5ec71129e7d33ad8f479edbcca

CITY DB SHA-256:
4aa3c3db9e1f6a6e3b6da9263c1d12a380ca0e33c9abf37309c1b213575458b8

ASN DB SHA-256:
839f74c8d161559f9dc7f414868bc0aff32f760141f58239ebbe9869733d034f

SOURCE SHAPE:
(58895, 24)

Unique source IPs: 1139

Running GeoLite2-City lookups...
Running GeoLite2-ASN lookups...
Lookups complete.

GEOIP V2 ENRICHMENT REPORT

Output shape: (58895, 39)

COVERAGE
Country       : 58867/58895 (99.95%)
Country Code  : 58867/58895 (99.95%)
City          : 47457/58895 (80.58%)
Latitude      : 58867/58895 (99.95%)
Longitude     : 58867/58895 (99.95%)
ASN           : 54048/58895 (91.77%)
ASN Org       : 54048/58895 (91.77%)

OLD vs NEW
Country matches    : 4704
Country mismatches : 50305
Country-code matches    : 4704
Country-code mismatches : 50305
ASN matches    : 296
ASN mismatches : 49953

TOP GEOIP COUNTRIES
geoip_country_v2
Australia          18493
United States   

In [ ]:
# STEP 2A — RAW SESSION RECONSTRUCTION AUDIT

import pandas as pd
import hashlib
import json
from pathlib import Path

SOURCE = "/content/unified_security_events_geoip_v2.csv"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=" * 70)
print("STEP 2A — RAW SESSION RECONSTRUCTION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# [1] SOURCE IDENTITY
# ------------------------------------------------------------

source_path = Path(SOURCE)

assert source_path.exists(), f"Source not found: {SOURCE}"

source_sha = sha256_file(SOURCE)

print("\n[1] SOURCE")
print("Path:", SOURCE)
print("SHA-256:", source_sha)

# ------------------------------------------------------------
# [2] LOAD
# ------------------------------------------------------------

df = pd.read_csv(SOURCE, low_memory=False)

print("\n[2] SHAPE")
print("Rows:", len(df))
print("Columns:", len(df.columns))

required = [
    "timestamp",
    "source_ip",
    "destination_ip",
    "source_port",
    "destination_port",
    "session_id",
    "honeypot",
    "protocol",
    "source_dataset"
]

missing_required = [c for c in required if c not in df.columns]

print("Missing required columns:", missing_required)

assert not missing_required

# ------------------------------------------------------------
# [3] TIMESTAMP VALIDATION
# ------------------------------------------------------------

df["_timestamp_parsed"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

invalid_timestamps = int(df["_timestamp_parsed"].isna().sum())

print("\n[3] TIMESTAMP")
print("Invalid timestamps:", invalid_timestamps)

# ------------------------------------------------------------
# [4] RAW SESSION PROFILE
# ------------------------------------------------------------

null_session = int(df["session_id"].isna().sum())
valid_session_rows = int(df["session_id"].notna().sum())

unique_sessions = int(df.loc[
    df["session_id"].notna(),
    "session_id"
].nunique())

print("\n[4] RAW SESSION PROFILE")
print("Null session_id:", null_session)
print("Valid session rows:", valid_session_rows)
print("Unique raw sessions:", unique_sessions)

# ------------------------------------------------------------
# [5] EVENTS PER SESSION
# ------------------------------------------------------------

session_counts = (
    df[df["session_id"].notna()]
    .groupby("session_id")
    .size()
)

print("\n[5] EVENTS PER RAW SESSION")
print("Min:", int(session_counts.min()))
print("Median:", float(session_counts.median()))
print("Mean:", float(session_counts.mean()))
print("Max:", int(session_counts.max()))

# ------------------------------------------------------------
# [6] MULTIPLE SOURCE IP CHECK
# ------------------------------------------------------------

session_ip_stats = (
    df[df["session_id"].notna()]
    .groupby("session_id")["source_ip"]
    .agg(
        unique_source_ips=lambda x: x.dropna().nunique(),
        null_source_ips=lambda x: x.isna().sum()
    )
)

multi_ip_sessions = session_ip_stats[
    session_ip_stats["unique_source_ips"] > 1
]

print("\n[6] MULTIPLE SOURCE IP SESSIONS")
print(
    "Sessions with >1 source_ip:",
    len(multi_ip_sessions)
)

print(
    "Rows belonging to these sessions:",
    int(
        df["session_id"].isin(
            multi_ip_sessions.index
        ).sum()
    )
)

if len(multi_ip_sessions) > 0:
    print("\nTop suspicious sessions:")
    print(
        multi_ip_sessions
        .sort_values("unique_source_ips", ascending=False)
        .head(20)
    )

# ------------------------------------------------------------
# [7] MULTIPLE DESTINATION IP CHECK
# ------------------------------------------------------------

session_dest_stats = (
    df[df["session_id"].notna()]
    .groupby("session_id")["destination_ip"]
    .nunique(dropna=True)
)

multi_dest_sessions = session_dest_stats[
    session_dest_stats > 1
]

print("\n[7] MULTIPLE DESTINATION IP SESSIONS")
print("Sessions with >1 destination_ip:", len(multi_dest_sessions))

# ------------------------------------------------------------
# [8] MULTIPLE HONEYPOT CHECK
# ------------------------------------------------------------

session_honeypot_stats = (
    df[df["session_id"].notna()]
    .groupby("session_id")["honeypot"]
    .nunique(dropna=True)
)

multi_honeypot_sessions = session_honeypot_stats[
    session_honeypot_stats > 1
]

print("\n[8] MULTIPLE HONEYPOT SESSIONS")
print("Sessions with >1 honeypot:", len(multi_honeypot_sessions))

# ------------------------------------------------------------
# [9] MULTIPLE PROTOCOL CHECK
# ------------------------------------------------------------

session_protocol_stats = (
    df[df["session_id"].notna()]
    .groupby("session_id")["protocol"]
    .nunique(dropna=True)
)

multi_protocol_sessions = session_protocol_stats[
    session_protocol_stats > 1
]

print("\n[9] MULTIPLE PROTOCOL SESSIONS")
print("Sessions with >1 protocol:", len(multi_protocol_sessions))

# ------------------------------------------------------------
# [10] MULTIPLE SOURCE DATASET CHECK
# ------------------------------------------------------------

session_source_stats = (
    df[df["session_id"].notna()]
    .groupby("session_id")["source_dataset"]
    .nunique(dropna=True)
)

multi_source_dataset_sessions = session_source_stats[
    session_source_stats > 1
]

print("\n[10] MULTIPLE SOURCE DATASET SESSIONS")
print(
    "Sessions with >1 source_dataset:",
    len(multi_source_dataset_sessions)
)

# ------------------------------------------------------------
# [11] TIMELINE CHECK
# ------------------------------------------------------------

session_time_stats = (
    df[df["session_id"].notna()]
    .groupby("session_id")["_timestamp_parsed"]
    .agg(
        first_seen="min",
        last_seen="max"
    )
)

session_time_stats["duration_seconds"] = (
    session_time_stats["last_seen"]
    - session_time_stats["first_seen"]
).dt.total_seconds()

negative_duration_sessions = session_time_stats[
    session_time_stats["duration_seconds"] < 0
]

print("\n[11] SESSION TIMELINE")
print(
    "Negative duration sessions:",
    len(negative_duration_sessions)
)

print(
    "Max duration seconds:",
    session_time_stats["duration_seconds"].max()
)

# ------------------------------------------------------------
# [12] CROSS-FIELD INCONSISTENCY
# ------------------------------------------------------------

cross_inconsistent = pd.DataFrame({
    "multi_source_ip": session_ip_stats["unique_source_ips"] > 1,
    "multi_destination_ip": session_dest_stats > 1,
    "multi_honeypot": session_honeypot_stats > 1,
    "multi_protocol": session_protocol_stats > 1,
    "multi_source_dataset": session_source_stats > 1
}).fillna(False)

cross_inconsistent["suspicious"] = cross_inconsistent.any(axis=1)

print("\n[12] SESSION CONSISTENCY")
print(
    "Sessions with at least one inconsistency:",
    int(cross_inconsistent["suspicious"].sum())
)

print("\nBreakdown:")
print(
    cross_inconsistent.drop(columns="suspicious")
    .sum()
)

# ------------------------------------------------------------
# [13] RAW SESSION SAMPLE
# ------------------------------------------------------------

suspicious_ids = cross_inconsistent[
    cross_inconsistent["suspicious"]
].index

if len(suspicious_ids) > 0:

    sample_cols = [
        "session_id",
        "timestamp",
        "source_ip",
        "destination_ip",
        "source_port",
        "destination_port",
        "honeypot",
        "protocol",
        "source_dataset"
    ]

    sample = (
        df[df["session_id"].isin(suspicious_ids)]
        [sample_cols]
        .sort_values(["session_id", "timestamp"])
        .head(100)
    )

    print("\n[13] SUSPICIOUS SESSION SAMPLE")
    print(sample.to_string(index=False))

# ------------------------------------------------------------
# [14] AUDIT SNAPSHOT
# ------------------------------------------------------------

audit = {
    "source": SOURCE,
    "source_sha256": source_sha,
    "rows": int(len(df)),
    "columns": int(len(df.columns) - 1),  # exclude temporary parsed timestamp
    "null_session_ids": null_session,
    "valid_session_rows": valid_session_rows,
    "unique_raw_sessions": unique_sessions,
    "invalid_timestamps": invalid_timestamps,
    "multi_source_ip_sessions": int(len(multi_ip_sessions)),
    "multi_destination_ip_sessions": int(len(multi_dest_sessions)),
    "multi_honeypot_sessions": int(len(multi_honeypot_sessions)),
    "multi_protocol_sessions": int(len(multi_protocol_sessions)),
    "multi_source_dataset_sessions": int(len(multi_source_dataset_sessions)),
    "negative_duration_sessions": int(len(negative_duration_sessions)),
    "sessions_with_any_inconsistency": int(
        cross_inconsistent["suspicious"].sum()
    )
}

print("\n" + "=" * 70)
print("STEP 2A AUDIT COMPLETE")
print("=" * 70)

print(json.dumps(audit, indent=2))

print("\nSOURCE FILE WAS NOT MODIFIED.")
print("NO SESSION RECONSTRUCTION PERFORMED.")
print("NO SESSION FEATURES CREATED.")
print("NO ML FEATURES CREATED.")

STEP 2A — RAW SESSION RECONSTRUCTION AUDIT

[1] SOURCE
Path: /content/unified_security_events_geoip_v2.csv
SHA-256: 601a841f544195063ec1b04c3657698a4a8124c8cc0f1b3d977f88207c109b19

[2] SHAPE
Rows: 58895
Columns: 39
Missing required columns: []

[3] TIMESTAMP
Invalid timestamps: 0

[4] RAW SESSION PROFILE
Null session_id: 75
Valid session rows: 58820
Unique raw sessions: 11879

[5] EVENTS PER RAW SESSION
Min: 1
Median: 5.0
Mean: 4.9515952521256
Max: 522

[6] MULTIPLE SOURCE IP SESSIONS
Sessions with >1 source_ip: 31
Rows belonging to these sessions: 2546

Top suspicious sessions:
                                      unique_source_ips  null_source_ips
session_id                                                              
ade88c93-30a2-4cd0-902c-9b7f99df6579                 44                0
3a7c3d75-c899-4e20-b2b5-59e5132468e6                 33                0
bb2155f4-c281-4351-9fa5-7c752d7f9aae                 31                0
be0c5be9-8d7b-4c65-8de1-d827ab6ced5c            

In [ ]:
# ============================================================
# STEP 2B — SESSION RECONSTRUCTION RULE TEST
# ============================================================

import pandas as pd
import hashlib
import json
from pathlib import Path

SOURCE = "/content/unified_security_events_geoip_v2.csv"

print("=" * 70)
print("STEP 2B — SESSION RECONSTRUCTION RULE TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load source
# ------------------------------------------------------------
df = pd.read_csv(SOURCE, low_memory=False)

print("\n[1] SOURCE")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# ------------------------------------------------------------
# 2. Parse timestamp
# ------------------------------------------------------------
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

print("\n[2] TIMESTAMP")
print("Invalid timestamps:", df["timestamp"].isna().sum())

# ------------------------------------------------------------
# 3. Required columns
# ------------------------------------------------------------
required = [
    "timestamp",
    "source_ip",
    "destination_ip",
    "honeypot",
    "protocol",
    "source_dataset",
    "session_id"
]

missing = [c for c in required if c not in df.columns]

print("\n[3] REQUIRED COLUMNS")
print("Missing:", missing)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 4. Basic reconstruction principle
#
# We DO NOT trust raw session_id blindly.
#
# Primary identity:
#   raw session_id
#
# Consistency checks:
#   source_ip
#   destination_ip
#   honeypot
#   protocol
#   source_dataset
#
# We will NOT split or merge sessions yet.
# This step only determines how many sessions are
# internally consistent vs suspicious.
# ------------------------------------------------------------

valid = df[df["session_id"].notna()].copy()

session_profile = (
    valid.groupby("session_id")
    .agg(
        events=("session_id", "size"),
        unique_source_ip=("source_ip", "nunique"),
        unique_destination_ip=("destination_ip", "nunique"),
        unique_honeypot=("honeypot", "nunique"),
        unique_protocol=("protocol", "nunique"),
        unique_source_dataset=("source_dataset", "nunique"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max")
    )
)

session_profile["duration_seconds"] = (
    session_profile["end_time"] -
    session_profile["start_time"]
).dt.total_seconds()

# ------------------------------------------------------------
# 5. Consistency flags
# ------------------------------------------------------------

session_profile["flag_multi_source_ip"] = (
    session_profile["unique_source_ip"] > 1
)

session_profile["flag_multi_destination_ip"] = (
    session_profile["unique_destination_ip"] > 1
)

session_profile["flag_multi_honeypot"] = (
    session_profile["unique_honeypot"] > 1
)

session_profile["flag_multi_protocol"] = (
    session_profile["unique_protocol"] > 1
)

session_profile["flag_multi_source_dataset"] = (
    session_profile["unique_source_dataset"] > 1
)

flag_columns = [
    "flag_multi_source_ip",
    "flag_multi_destination_ip",
    "flag_multi_honeypot",
    "flag_multi_protocol",
    "flag_multi_source_dataset"
]

session_profile["suspicious"] = session_profile[flag_columns].any(axis=1)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\n[4] SESSION COUNTS")
print("Total valid raw sessions:", len(session_profile))
print(
    "Consistent sessions:",
    (~session_profile["suspicious"]).sum()
)
print(
    "Suspicious sessions:",
    session_profile["suspicious"].sum()
)

print("\n[5] INCONSISTENCY BREAKDOWN")

for col in flag_columns:
    print(
        col,
        ":",
        int(session_profile[col].sum())
    )

# ------------------------------------------------------------
# 7. Rows affected
# ------------------------------------------------------------

suspicious_ids = session_profile.index[
    session_profile["suspicious"]
]

suspicious_rows = valid[
    valid["session_id"].isin(suspicious_ids)
]

print("\n[6] ROW IMPACT")
print(
    "Rows belonging to suspicious sessions:",
    len(suspicious_rows)
)

print(
    "Rows belonging to consistent sessions:",
    len(valid) - len(suspicious_rows)
)

# ------------------------------------------------------------
# 8. Time sanity
# ------------------------------------------------------------

print("\n[7] TIMELINE")
print(
    "Negative durations:",
    (session_profile["duration_seconds"] < 0).sum()
)

print(
    "Maximum duration seconds:",
    session_profile["duration_seconds"].max()
)

# ------------------------------------------------------------
# 9. IMPORTANT:
#    DO NOT CREATE RECONSTRUCTED SESSION IDS YET.
# ------------------------------------------------------------

print("\n[8] RECONSTRUCTION STATUS")
print("Raw session IDs preserved.")
print("No session IDs changed.")
print("No sessions merged.")
print("No sessions split.")
print("No output dataset created.")
print("No session features created.")
print("No ML features created.")

print("\n" + "=" * 70)
print("STEP 2B TEST COMPLETE")
print("=" * 70)

STEP 2B — SESSION RECONSTRUCTION RULE TEST

[1] SOURCE
Rows: 58895
Columns: 39

[2] TIMESTAMP
Invalid timestamps: 0

[3] REQUIRED COLUMNS
Missing: []

[4] SESSION COUNTS
Total valid raw sessions: 11879
Consistent sessions: 11832
Suspicious sessions: 47

[5] INCONSISTENCY BREAKDOWN
flag_multi_source_ip : 31
flag_multi_destination_ip : 16
flag_multi_honeypot : 0
flag_multi_protocol : 0
flag_multi_source_dataset : 0

[6] ROW IMPACT
Rows belonging to suspicious sessions: 2655
Rows belonging to consistent sessions: 56165

[7] TIMELINE
Negative durations: 0
Maximum duration seconds: 565592.228657484

[8] RECONSTRUCTION STATUS
Raw session IDs preserved.
No session IDs changed.
No sessions merged.
No sessions split.
No output dataset created.
No session features created.
No ML features created.

STEP 2B TEST COMPLETE


In [ ]:
# ============================================================
# STEP 2C — SUSPICIOUS SESSION DEEP AUDIT
# ============================================================

import pandas as pd

SOURCE = "/content/unified_security_events_geoip_v2.csv"

print("=" * 70)
print("STEP 2C — SUSPICIOUS SESSION DEEP AUDIT")
print("=" * 70)

df = pd.read_csv(SOURCE, low_memory=False)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True,
    errors="coerce"
)

valid = df[df["session_id"].notna()].copy()

# ------------------------------------------------------------
# Build session-level profile
# ------------------------------------------------------------

profile = (
    valid.groupby("session_id")
    .agg(
        events=("session_id", "size"),
        unique_source_ip=("source_ip", "nunique"),
        unique_destination_ip=("destination_ip", "nunique"),
        unique_honeypot=("honeypot", "nunique"),
        unique_protocol=("protocol", "nunique"),
        unique_source_dataset=("source_dataset", "nunique"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max")
    )
)

profile["duration_seconds"] = (
    profile["end_time"] - profile["start_time"]
).dt.total_seconds()

profile["multi_source_ip"] = (
    profile["unique_source_ip"] > 1
)

profile["multi_destination_ip"] = (
    profile["unique_destination_ip"] > 1
)

profile["suspicious"] = (
    profile["multi_source_ip"] |
    profile["multi_destination_ip"]
)

suspicious_ids = profile.index[profile["suspicious"]]

# ------------------------------------------------------------
# Analyze suspicious sessions
# ------------------------------------------------------------

sus = valid[valid["session_id"].isin(suspicious_ids)].copy()

sus = sus.sort_values(
    ["session_id", "timestamp"]
)

sus["gap_seconds"] = (
    sus.groupby("session_id")["timestamp"]
    .diff()
    .dt.total_seconds()
)

# ------------------------------------------------------------
# Session-level summary
# ------------------------------------------------------------

summary = (
    sus.groupby("session_id")
    .agg(
        events=("session_id", "size"),
        source_ips=("source_ip", "nunique"),
        destination_ips=("destination_ip", "nunique"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        max_gap_seconds=("gap_seconds", "max"),
        median_gap_seconds=("gap_seconds", "median"),
        mean_gap_seconds=("gap_seconds", "mean")
    )
)

summary["duration_seconds"] = (
    summary["end_time"] - summary["start_time"]
).dt.total_seconds()

summary["reason"] = summary.apply(
    lambda r:
        "multi_source_ip + multi_destination_ip"
        if r["source_ips"] > 1 and r["destination_ips"] > 1
        else "multi_source_ip"
        if r["source_ips"] > 1
        else "multi_destination_ip",
    axis=1
)

print("\n[1] SUSPICIOUS SESSION SUMMARY")
print("Suspicious sessions:", len(summary))
print("Rows:", len(sus))

print("\nReason breakdown:")
print(summary["reason"].value_counts())

print("\n[2] SUMMARY STATISTICS")
print(
    summary[
        [
            "events",
            "source_ips",
            "destination_ips",
            "duration_seconds",
            "max_gap_seconds"
        ]
    ].describe()
)

# ------------------------------------------------------------
# Print every suspicious session
# ------------------------------------------------------------

print("\n[3] ALL SUSPICIOUS SESSIONS")
print("-" * 70)

for sid, row in summary.sort_values(
    ["reason", "events"],
    ascending=[True, False]
).iterrows():

    print("\nSESSION:", sid)
    print("Reason:", row["reason"])
    print("Events:", int(row["events"]))
    print("Source IPs:", int(row["source_ips"]))
    print("Destination IPs:", int(row["destination_ips"]))
    print("Start:", row["start_time"])
    print("End:", row["end_time"])
    print("Duration:", row["duration_seconds"])
    print("Max gap:", row["max_gap_seconds"])

    session_rows = sus[sus["session_id"] == sid]

    print("\nSource IP sequence:")
    print(
        session_rows[
            ["timestamp", "source_ip", "destination_ip"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2C COMPLETE")
print("=" * 70)

print("NO SESSION IDS CHANGED.")
print("NO SESSIONS SPLIT.")
print("NO SESSIONS MERGED.")
print("NO OUTPUT DATASET CREATED.")
print("NO SESSION FEATURES CREATED.")
print("NO ML FEATURES CREATED.")

STEP 2C — SUSPICIOUS SESSION DEEP AUDIT

[1] SUSPICIOUS SESSION SUMMARY
Suspicious sessions: 47
Rows: 2655

Reason breakdown:
reason
multi_source_ip         31
multi_destination_ip    16
Name: count, dtype: int64

[2] SUMMARY STATISTICS
           events  source_ips  destination_ips  duration_seconds  \
count   47.000000   47.000000        47.000000         47.000000   
mean    56.489362   14.042553         0.680851      51312.976574   
std     96.125794   12.356933         0.957950      84324.169773   
min      2.000000    1.000000         0.000000          0.694776   
25%      7.000000    1.000000         0.000000         10.454176   
50%     22.000000   14.000000         0.000000      37860.942347   
75%     39.000000   25.000000         2.000000      77089.919380   
max    522.000000   44.000000         2.000000     565592.228657   

       max_gap_seconds  
count        47.000000  
mean      19999.877706  
std       81851.232853  
min           0.349563  
25%           7.945516  


In [ ]:
# ============================================================
# STEP 2C — RECONSTRUCTION CANDIDATE GENERATION
# ============================================================

import pandas as pd
import numpy as np
import json
import hashlib
from pathlib import Path

SOURCE = "/content/unified_security_events_geoip_v2.csv"

print("=" * 70)
print("STEP 2C — RECONSTRUCTION CANDIDATE GENERATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD SOURCE
# ------------------------------------------------------------

df = pd.read_csv(SOURCE, low_memory=False)

print("\n[1] SOURCE")
print("Path:", SOURCE)
print("Shape:", df.shape)

# ------------------------------------------------------------
# 2. REQUIRED COLUMNS
# ------------------------------------------------------------

required = [
    "session_id",
    "timestamp",
    "source_ip",
    "destination_ip",
    "honeypot",
    "protocol",
    "source_dataset",
]

missing = [c for c in required if c not in df.columns]

print("\n[2] REQUIRED COLUMNS")
print("Missing:", missing)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 3. TIMESTAMP
# ------------------------------------------------------------

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

invalid_ts = int(df["timestamp"].isna().sum())

print("\n[3] TIMESTAMP")
print("Invalid timestamps:", invalid_ts)

if invalid_ts:
    raise ValueError("Invalid timestamps detected.")

# ------------------------------------------------------------
# 4. KEEP ONLY VALID RAW SESSIONS
# ------------------------------------------------------------

valid = df[df["session_id"].notna()].copy()

print("\n[4] VALID RAW SESSION ROWS")
print("Rows:", len(valid))
print("Unique sessions:", valid["session_id"].nunique())

# ------------------------------------------------------------
# 5. BUILD SESSION PROFILE
# ------------------------------------------------------------

profiles = []

for sid, g in valid.groupby("session_id", sort=False):

    g = g.sort_values("timestamp").copy()

    timestamps = g["timestamp"]

    start = timestamps.min()
    end = timestamps.max()

    duration = (end - start).total_seconds()

    gaps = timestamps.diff().dt.total_seconds().dropna()

    max_gap = float(gaps.max()) if len(gaps) else 0.0

    source_ips = g["source_ip"].dropna().unique().tolist()
    destination_ips = g["destination_ip"].dropna().unique().tolist()
    honeypots = g["honeypot"].dropna().unique().tolist()
    protocols = g["protocol"].dropna().unique().tolist()
    datasets = g["source_dataset"].dropna().unique().tolist()

    flags = []

    if len(source_ips) > 1:
        flags.append("multi_source_ip")

    if len(destination_ips) > 1:
        flags.append("multi_destination_ip")

    if len(honeypots) > 1:
        flags.append("multi_honeypot")

    if len(protocols) > 1:
        flags.append("multi_protocol")

    if len(datasets) > 1:
        flags.append("multi_source_dataset")

    if flags:

        profiles.append({
            "raw_session_id": sid,
            "events": len(g),
            "source_ips": len(source_ips),
            "destination_ips": len(destination_ips),
            "honeypots": len(honeypots),
            "protocols": len(protocols),
            "source_datasets": len(datasets),
            "start_time": start,
            "end_time": end,
            "duration_seconds": duration,
            "max_gap_seconds": max_gap,
            "reason": "|".join(flags),
            "source_ip_values": json.dumps(
                [str(x) for x in source_ips]
            ),
            "destination_ip_values": json.dumps(
                [str(x) for x in destination_ips]
            ),
        })

candidates = pd.DataFrame(profiles)

# ------------------------------------------------------------
# 6. SAFETY CHECK
# ------------------------------------------------------------

print("\n[5] SUSPICIOUS SESSION CANDIDATES")
print("Count:", len(candidates))

if len(candidates) != 47:
    print("WARNING: Expected 47 suspicious sessions.")
    print("Actual:", len(candidates))

# ------------------------------------------------------------
# 7. RECONSTRUCTION STATUS
# ------------------------------------------------------------

def classify(row):

    reasons = set(row["reason"].split("|"))

    # Multiple source IPs are inherently ambiguous.
    if "multi_source_ip" in reasons:
        return "AMBIGUOUS"

    # Multiple destinations may be legitimate within one interaction.
    if "multi_destination_ip" in reasons:
        return "REVIEW_REQUIRED"

    return "REVIEW_REQUIRED"


candidates["reconstruction_status"] = candidates.apply(
    classify,
    axis=1
)

# ------------------------------------------------------------
# 8. IMPORTANT:
#    NO SESSION ID IS GENERATED YET
# ------------------------------------------------------------

candidates["proposed_session_id"] = pd.NA

# No merge.
# No split.
# No session_id replacement.
# No event modification.

# ------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------

print("\n[6] REASON BREAKDOWN")
print(
    candidates["reason"]
    .value_counts()
)

print("\n[7] RECONSTRUCTION STATUS")
print(
    candidates["reconstruction_status"]
    .value_counts()
)

print("\n[8] DURATION SUMMARY")
print(
    candidates[
        [
            "events",
            "source_ips",
            "destination_ips",
            "duration_seconds",
            "max_gap_seconds"
        ]
    ].describe()
)

# ------------------------------------------------------------
# 10. SHOW CANDIDATES
# ------------------------------------------------------------

print("\n[9] CANDIDATE TABLE")
display(
    candidates[
        [
            "raw_session_id",
            "reason",
            "events",
            "source_ips",
            "destination_ips",
            "start_time",
            "end_time",
            "duration_seconds",
            "max_gap_seconds",
            "reconstruction_status"
        ]
    ].sort_values(
        ["reconstruction_status", "duration_seconds"],
        ascending=[True, False]
    )
)

# ------------------------------------------------------------
# 11. OUTPUT — AUDIT ONLY
# ------------------------------------------------------------

OUTPUT = "/content/session_reconstruction_candidates_v1.csv"
AUDIT = "/content/session_reconstruction_candidates_v1_audit.json"

candidates.to_csv(OUTPUT, index=False)

# ------------------------------------------------------------
# 12. HASH OUTPUT
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

output_sha = sha256_file(OUTPUT)

# ------------------------------------------------------------
# 13. SOURCE HASH
# ------------------------------------------------------------

source_sha = sha256_file(SOURCE)

audit = {
    "step": "2C",
    "source": SOURCE,
    "source_sha256": source_sha,
    "source_rows": int(len(df)),
    "valid_session_rows": int(len(valid)),
    "suspicious_sessions": int(len(candidates)),
    "expected_suspicious_sessions": 47,
    "reason_counts": {
        str(k): int(v)
        for k, v in candidates["reason"].value_counts().items()
    },
    "status_counts": {
        str(k): int(v)
        for k, v in candidates["reconstruction_status"].value_counts().items()
    },
    "output": OUTPUT,
    "output_sha256": output_sha,
    "raw_session_ids_preserved": True,
    "proposed_session_ids_created": False,
    "sessions_merged": False,
    "sessions_split": False,
    "source_modified": False,
    "event_data_modified": False,
    "session_features_created": False,
    "ml_features_created": False
}

with open(AUDIT, "w") as f:
    json.dump(audit, f, indent=2)

# ------------------------------------------------------------
# 14. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2C COMPLETE")
print("=" * 70)

print("Candidate output:")
print(OUTPUT)

print("Audit:")
print(AUDIT)

print("Output SHA-256:")
print(output_sha)

print("\nSOURCE FILE WAS NOT MODIFIED.")
print("NO SESSION IDs WERE CHANGED.")
print("NO SESSIONS WERE MERGED.")
print("NO SESSIONS WERE SPLIT.")
print("NO SESSION FEATURES CREATED.")
print("NO ML FEATURES CREATED.")

STEP 2C — RECONSTRUCTION CANDIDATE GENERATION

[1] SOURCE
Path: /content/unified_security_events_geoip_v2.csv
Shape: (58895, 39)

[2] REQUIRED COLUMNS
Missing: []

[3] TIMESTAMP
Invalid timestamps: 0

[4] VALID RAW SESSION ROWS
Rows: 58820
Unique sessions: 11879

[5] SUSPICIOUS SESSION CANDIDATES
Count: 47

[6] REASON BREAKDOWN
reason
multi_source_ip         31
multi_destination_ip    16
Name: count, dtype: int64

[7] RECONSTRUCTION STATUS
reconstruction_status
AMBIGUOUS          31
REVIEW_REQUIRED    16
Name: count, dtype: int64

[8] DURATION SUMMARY
           events  source_ips  destination_ips  duration_seconds  \
count   47.000000   47.000000        47.000000         47.000000   
mean    56.489362   14.042553         0.680851      51312.976574   
std     96.125794   12.356933         0.957950      84324.169773   
min      2.000000    1.000000         0.000000          0.694776   
25%      7.000000    1.000000         0.000000         10.454176   
50%     22.000000   14.000000     

,raw_session_id,reason,events,source_ips,destination_ips,start_time,end_time,duration_seconds,max_gap_seconds,reconstruction_status
46,5,multi_source_ip,2,2,0,2020-02-10 03:39:50.386400700+00:00,2020-02-16 16:46:22.615058184+00:00,565592.228657,565592.228657,AMBIGUOUS
33,44a69b7d-2d33-4194-b2f7-029947810fad,multi_source_ip,36,29,0,2020-02-12 03:55:01.994487+00:00,2020-02-13 03:23:00.718838+00:00,84478.724351,12261.378118,AMBIGUOUS
18,3a7c3d75-c899-4e20-b2b5-59e5132468e6,multi_source_ip,273,33,0,2020-01-24 03:50:00.150430+00:00,2020-01-25 03:14:04.014699+00:00,84243.864269,11713.100895,AMBIGUOUS
23,54259087-aa22-492f-852a-434e4852377b,multi_source_ip,36,22,0,2020-02-03 03:46:44.741871+00:00,2020-02-04 03:07:22.831741+00:00,84038.089870,14987.640672,AMBIGUOUS
28,beba058b-e1f6-413b-b5cb-2f5f102008bd,multi_source_ip,30,26,0,2020-02-08 03:54:17.485975+00:00,2020-02-09 03:14:51.804126+00:00,84034.318151,13158.467370,AMBIGUOUS
36,f863db4c-8829-41a4-9dfa-8e6d737465bf,multi_source_ip,270,20,0,2020-02-15 03:54:06.885966+00:00,2020-02-16 03:02:19.161857+00:00,83292.275891,15895.866383,AMBIGUOUS
19,be0c5be9-8d7b-4c65-8de1-d827ab6ced5c,multi_source_ip,175,31,0,2020-01-25 03:53:40.850996+00:00,2020-01-26 02:55:34.658536+00:00,82913.807540,23099.908426,AMBIGUOUS
31,b10ea748-af60-4824-be0d-cda9833ce73f,multi_source_ip,27,27,0,2020-02-10 04:09:55.686066+00:00,2020-02-11 03:03:09.936014+00:00,82394.249948,8663.217594,AMBIGUOUS
39,bb2155f4-c281-4351-9fa5-7c752d7f9aae,multi_source_ip,37,31,0,2020-02-17 03:41:19.162081+00:00,2020-02-18 01:53:58.169799+00:00,79959.007718,9983.082935,AMBIGUOUS
35,5937b960-0ccd-4460-9a6e-31596778f1ea,multi_source_ip,38,23,0,2020-02-14 05:05:26.332743+00:00,2020-02-15 03:01:30.873916+00:00,78964.541173,14247.550661,AMBIGUOUS



STEP 2C COMPLETE
Candidate output:
/content/session_reconstruction_candidates_v1.csv
Audit:
/content/session_reconstruction_candidates_v1_audit.json
Output SHA-256:
96339e4889acc8b24bfd069e88a59b6eaf4913a6d9101cca7fe39972f318b306

SOURCE FILE WAS NOT MODIFIED.
NO SESSION IDs WERE CHANGED.
NO SESSIONS WERE MERGED.
NO SESSIONS WERE SPLIT.
NO SESSION FEATURES CREATED.
NO ML FEATURES CREATED.


In [ ]:
# ============================================================
# STEP 2D — SESSION RECONSTRUCTION RULE VALIDATION
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import json

SOURCE = "/content/unified_security_events_geoip_v2.csv"
CANDIDATES = "/content/session_reconstruction_candidates_v1.csv"

print("=" * 70)
print("STEP 2D — SESSION RECONSTRUCTION RULE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_csv(SOURCE, low_memory=False)
cand = pd.read_csv(CANDIDATES, low_memory=False)

print("\n[1] INPUT")
print("Source shape:", df.shape)
print("Candidate shape:", cand.shape)

# ------------------------------------------------------------
# 2. REQUIRED COLUMNS
# ------------------------------------------------------------

required_source = [
    "session_id",
    "timestamp",
    "source_ip",
    "destination_ip",
    "honeypot",
    "protocol",
    "source_dataset"
]

required_candidates = [
    "raw_session_id",
    "reason",
    "events",
    "source_ips",
    "destination_ips",
    "start_time",
    "end_time",
    "duration_seconds",
    "max_gap_seconds",
    "reconstruction_status"
]

missing_source = [
    c for c in required_source
    if c not in df.columns
]

missing_candidates = [
    c for c in required_candidates
    if c not in cand.columns
]

print("\n[2] REQUIRED COLUMNS")
print("Missing source:", missing_source)
print("Missing candidates:", missing_candidates)

if missing_source:
    raise ValueError(
        f"Source missing required columns: {missing_source}"
    )

if missing_candidates:
    raise ValueError(
        f"Candidates missing required columns: {missing_candidates}"
    )

# ------------------------------------------------------------
# 3. TIMESTAMP VALIDATION
# ------------------------------------------------------------

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

cand["start_time"] = pd.to_datetime(
    cand["start_time"],
    errors="coerce",
    utc=True
)

cand["end_time"] = pd.to_datetime(
    cand["end_time"],
    errors="coerce",
    utc=True
)

invalid_source_ts = int(df["timestamp"].isna().sum())
invalid_candidate_ts = int(
    cand[["start_time", "end_time"]].isna().any(axis=1).sum()
)

print("\n[3] TIMESTAMP")
print("Invalid source timestamps:", invalid_source_ts)
print("Invalid candidate timestamps:", invalid_candidate_ts)

if invalid_source_ts != 0:
    raise ValueError("Source contains invalid timestamps.")

if invalid_candidate_ts != 0:
    raise ValueError("Candidate timestamps are invalid.")

# ------------------------------------------------------------
# 4. CANDIDATE COUNT
# ------------------------------------------------------------

print("\n[4] CANDIDATE COUNT")
print("Candidates:", len(cand))

if len(cand) != 47:
    raise ValueError(
        f"Expected 47 suspicious sessions, found {len(cand)}"
    )

# ------------------------------------------------------------
# 5. UNIQUE RAW SESSION IDS
# ------------------------------------------------------------

unique_candidates = cand["raw_session_id"].nunique()

print("\n[5] RAW SESSION ID UNIQUENESS")
print("Rows:", len(cand))
print("Unique raw session IDs:", unique_candidates)

if unique_candidates != len(cand):
    raise ValueError(
        "Duplicate raw_session_id detected in candidates."
    )

# ------------------------------------------------------------
# 6. VERIFY CANDIDATES EXIST IN SOURCE
# ------------------------------------------------------------

source_session_ids = set(
    df.loc[df["session_id"].notna(), "session_id"]
    .astype(str)
)

candidate_session_ids = set(
    cand["raw_session_id"].astype(str)
)

missing_from_source = (
    candidate_session_ids - source_session_ids
)

print("\n[6] SOURCE MEMBERSHIP")
print(
    "Candidate IDs missing from source:",
    len(missing_from_source)
)

if missing_from_source:
    raise ValueError(
        "Some candidate session IDs do not exist in source."
    )

# ------------------------------------------------------------
# 7. REASON VALIDATION
# ------------------------------------------------------------

valid_reasons = {
    "multi_source_ip",
    "multi_destination_ip"
}

observed_reasons = set()

for value in cand["reason"].dropna():
    for reason in str(value).split("|"):
        observed_reasons.add(reason)

unexpected_reasons = observed_reasons - valid_reasons

print("\n[7] REASON VALIDATION")
print("Observed:", sorted(observed_reasons))
print("Unexpected:", sorted(unexpected_reasons))

if unexpected_reasons:
    raise ValueError(
        f"Unexpected reconstruction reasons: {unexpected_reasons}"
    )

# ------------------------------------------------------------
# 8. EXPECTED BREAKDOWN
# ------------------------------------------------------------

reason_counts = cand["reason"].value_counts()

multi_source_count = int(
    reason_counts.get("multi_source_ip", 0)
)

multi_destination_count = int(
    reason_counts.get("multi_destination_ip", 0)
)

print("\n[8] REASON BREAKDOWN")
print("multi_source_ip:", multi_source_count)
print("multi_destination_ip:", multi_destination_count)

if multi_source_count != 31:
    raise ValueError(
        f"Expected 31 multi_source_ip cases, got {multi_source_count}"
    )

if multi_destination_count != 16:
    raise ValueError(
        f"Expected 16 multi_destination_ip cases, got {multi_destination_count}"
    )

# ------------------------------------------------------------
# 9. RECONSTRUCTION SAFETY RULE
#
# No automatic reconstruction is authorized yet.
#
# multi_source_ip:
#     ALWAYS AMBIGUOUS
#
# multi_destination_ip:
#     REVIEW_REQUIRED
#
# No merge/split decision is made in 2D.
# ------------------------------------------------------------

cand["validated_rule"] = np.where(
    cand["reason"].eq("multi_source_ip"),
    "HOLD_AMBIGUOUS",
    np.where(
        cand["reason"].eq("multi_destination_ip"),
        "HOLD_REVIEW",
        "HOLD"
    )
)

# ------------------------------------------------------------
# 10. VERIFY NO AUTOMATIC SESSION ID
# ------------------------------------------------------------

if "proposed_session_id" in cand.columns:
    proposed_non_null = int(
        cand["proposed_session_id"].notna().sum()
    )
else:
    proposed_non_null = 0

print("\n[9] PROPOSED SESSION IDS")
print("Non-null proposed IDs:", proposed_non_null)

if proposed_non_null != 0:
    raise ValueError(
        "Proposed session IDs already exist. "
        "2D must not create reconstructed IDs."
    )

# ------------------------------------------------------------
# 11. VERIFY SOURCE ROW COUNT
# ------------------------------------------------------------

source_rows = len(df)

print("\n[10] SOURCE SAFETY")
print("Source rows:", source_rows)

if source_rows != 58895:
    raise ValueError(
        f"Unexpected source row count: {source_rows}"
    )

# ------------------------------------------------------------
# 12. FINAL RULE TABLE
# ------------------------------------------------------------

print("\n[11] VALIDATED RULES")

rule_table = pd.DataFrame({
    "condition": [
        "multi_source_ip",
        "multi_destination_ip"
    ],
    "count": [
        multi_source_count,
        multi_destination_count
    ],
    "rule": [
        "HOLD — AMBIGUOUS; no automatic split",
        "HOLD — REVIEW_REQUIRED; no automatic merge"
    ]
})

display(rule_table)

# ------------------------------------------------------------
# 13. AUDIT
# ------------------------------------------------------------

AUDIT = "/content/session_reconstruction_step2d_audit.json"

audit = {
    "step": "2D",
    "source": SOURCE,
    "candidate_file": CANDIDATES,
    "source_rows": int(len(df)),
    "source_columns": int(len(df.columns)),
    "candidate_rows": int(len(cand)),
    "candidate_unique_raw_sessions": int(unique_candidates),
    "multi_source_ip": multi_source_count,
    "multi_destination_ip": multi_destination_count,
    "invalid_source_timestamps": invalid_source_ts,
    "invalid_candidate_timestamps": invalid_candidate_ts,
    "unexpected_reasons": sorted(unexpected_reasons),
    "automatic_reconstruction_performed": False,
    "sessions_merged": False,
    "sessions_split": False,
    "session_ids_changed": False,
    "source_modified": False,
    "session_features_created": False,
    "ml_features_created": False,
    "rule_status": "PASS"
}

with open(AUDIT, "w") as f:
    json.dump(audit, f, indent=2)

# ------------------------------------------------------------
# 14. HASH AUDIT
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

audit_sha = sha256_file(AUDIT)

# ------------------------------------------------------------
# 15. FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2D — PASS")
print("=" * 70)

print("Audit:")
print(AUDIT)

print("Audit SHA-256:")
print(audit_sha)

print("\nRULE:")
print("31 multi_source_ip      -> HOLD / AMBIGUOUS")
print("16 multi_destination_ip -> HOLD / REVIEW_REQUIRED")

print("\nNO RECONSTRUCTION PERFORMED.")
print("NO SESSION IDS CHANGED.")
print("NO SESSIONS MERGED.")
print("NO SESSIONS SPLIT.")
print("SOURCE FILE WAS NOT MODIFIED.")
print("NO SESSION FEATURES CREATED.")
print("NO ML FEATURES CREATED.")

STEP 2D — SESSION RECONSTRUCTION RULE VALIDATION

[1] INPUT
Source shape: (58895, 39)
Candidate shape: (47, 16)

[2] REQUIRED COLUMNS
Missing source: []
Missing candidates: []

[3] TIMESTAMP
Invalid source timestamps: 0
Invalid candidate timestamps: 0

[4] CANDIDATE COUNT
Candidates: 47

[5] RAW SESSION ID UNIQUENESS
Rows: 47
Unique raw session IDs: 47

[6] SOURCE MEMBERSHIP
Candidate IDs missing from source: 0

[7] REASON VALIDATION
Observed: ['multi_destination_ip', 'multi_source_ip']
Unexpected: []

[8] REASON BREAKDOWN
multi_source_ip: 31
multi_destination_ip: 16

[9] PROPOSED SESSION IDS
Non-null proposed IDs: 0

[10] SOURCE SAFETY
Source rows: 58895

[11] VALIDATED RULES


,condition,count,rule
0,multi_source_ip,31,HOLD — AMBIGUOUS; no automatic split
1,multi_destination_ip,16,HOLD — REVIEW_REQUIRED; no automatic merge



STEP 2D — PASS
Audit:
/content/session_reconstruction_step2d_audit.json
Audit SHA-256:
938c7cc70fe10ea566b4a2141366af72b1133fc05a290aff6de3789abc4ea9ff

RULE:
31 multi_source_ip      -> HOLD / AMBIGUOUS
16 multi_destination_ip -> HOLD / REVIEW_REQUIRED

NO RECONSTRUCTION PERFORMED.
NO SESSION IDS CHANGED.
NO SESSIONS MERGED.
NO SESSIONS SPLIT.
SOURCE FILE WAS NOT MODIFIED.
NO SESSION FEATURES CREATED.
NO ML FEATURES CREATED.


In [ ]:
# ============================================================
# STEP 2E — SESSION RECONSTRUCTION EXECUTION
# SAFE / AUDITABLE / NO RAW SOURCE MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
from datetime import datetime, timezone

# ------------------------------------------------------------
# 0) PATHS
# ------------------------------------------------------------

SOURCE = Path("/content/unified_security_events_geoip_v2.csv")
CANDIDATES = Path("/content/session_reconstruction_candidates_v1.csv")

OUT_DIR = Path("/content/session_reconstruction_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RECONSTRUCTED = OUT_DIR / "session_reconstructed_v1.csv"
AUDIT = OUT_DIR / "session_reconstructed_v1_audit.json"
HASH_FILE = OUT_DIR / "session_reconstructed_v1.sha256"

# ------------------------------------------------------------
# 1) HASH FUNCTION
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# ------------------------------------------------------------
# 2) LOAD
# ------------------------------------------------------------

assert SOURCE.exists(), f"Source not found: {SOURCE}"
assert CANDIDATES.exists(), f"Candidates not found: {CANDIDATES}"

df = pd.read_csv(SOURCE)
cand = pd.read_csv(CANDIDATES)

print("SOURCE:", df.shape)
print("CANDIDATES:", cand.shape)

# ------------------------------------------------------------
# 3) REQUIRED COLUMNS
# ------------------------------------------------------------

required_source = {
    "session_id",
    "timestamp",
    "source_ip",
    "destination_ip"
}

missing_source = sorted(required_source - set(df.columns))

required_candidates = {
    "raw_session_id",
    "reason",
    "reconstruction_status"
}

missing_candidates = sorted(required_candidates - set(cand.columns))

assert not missing_source, f"Missing source columns: {missing_source}"
assert not missing_candidates, f"Missing candidate columns: {missing_candidates}"

# ------------------------------------------------------------
# 4) TIMESTAMP VALIDATION
# ------------------------------------------------------------

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

invalid_ts = int(df["timestamp"].isna().sum())

assert invalid_ts == 0, f"Invalid timestamps found: {invalid_ts}"

# ------------------------------------------------------------
# 5) IMPORTANT:
#    DO NOT AUTOMATICALLY SPLIT / MERGE THE 47 CASES
# ------------------------------------------------------------

reason_counts = cand["reason"].value_counts().to_dict()

multi_source = int(reason_counts.get("multi_source_ip", 0))
multi_destination = int(reason_counts.get("multi_destination_ip", 0))

assert multi_source == 31, f"Expected 31 multi_source_ip, got {multi_source}"
assert multi_destination == 16, f"Expected 16 multi_destination_ip, got {multi_destination}"

# ------------------------------------------------------------
# 6) SAFE RECONSTRUCTION RULE
#
# Every raw event keeps its original session_id as provenance.
# No automatic split.
# No automatic merge.
#
# We create an explicit reconstructed_session_id:
#     reconstructed_<raw_session_id>
#
# This is NOT changing the raw ID.
# ------------------------------------------------------------

df["raw_session_id"] = df["session_id"].astype(str)

df["reconstructed_session_id"] = (
    "reconstructed_" + df["raw_session_id"]
)

# ------------------------------------------------------------
# 7) RECONSTRUCTION DECISION LABEL
# ------------------------------------------------------------

candidate_map = (
    cand[
        [
            "raw_session_id",
            "reason",
            "reconstruction_status"
        ]
    ]
    .drop_duplicates("raw_session_id")
    .copy()
)

candidate_map["raw_session_id"] = (
    candidate_map["raw_session_id"].astype(str)
)

df = df.merge(
    candidate_map,
    on="raw_session_id",
    how="left",
    validate="many_to_one"
)

df["reconstruction_action"] = np.where(
    df["reason"].eq("multi_source_ip"),
    "HOLD_AMBIGUOUS_NO_SPLIT",
    np.where(
        df["reason"].eq("multi_destination_ip"),
        "HOLD_REVIEW_REQUIRED_NO_MERGE",
        "UNCHANGED"
    )
)

# ------------------------------------------------------------
# 8) ORDER EVENTS FOR REPLAY
# ------------------------------------------------------------

df = df.sort_values(
    ["reconstructed_session_id", "timestamp"],
    kind="stable"
).reset_index(drop=True)

# Event sequence inside reconstructed session
df["event_sequence"] = (
    df.groupby("reconstructed_session_id", sort=False)
      .cumcount() + 1
)

# ------------------------------------------------------------
# 9) SAFETY CHECKS
# ------------------------------------------------------------

source_rows = len(df)

assert source_rows == len(
    pd.read_csv(SOURCE)
), "ROW COUNT CHANGED"

assert df["raw_session_id"].notna().all()
assert df["reconstructed_session_id"].notna().all()

# Every original raw session maps to exactly one reconstructed session
mapping_check = (
    df.groupby("raw_session_id")["reconstructed_session_id"]
      .nunique()
)

assert mapping_check.max() == 1, (
    "A raw session was split unexpectedly."
)

# No reconstruction merges
reverse_check = (
    df.groupby("reconstructed_session_id")["raw_session_id"]
      .nunique()
)

assert reverse_check.max() == 1, (
    "Multiple raw sessions were merged unexpectedly."
)

# ------------------------------------------------------------
# 10) SAVE RECONSTRUCTED DATASET
# ------------------------------------------------------------

df.to_csv(RECONSTRUCTED, index=False)

# ------------------------------------------------------------
# 11) AUDIT
# ------------------------------------------------------------

audit = {
    "step": "2E",
    "title": "SESSION RECONSTRUCTION EXECUTION",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "source": str(SOURCE),
    "candidate_input": str(CANDIDATES),
    "output": str(RECONSTRUCTED),

    "source_rows": int(source_rows),
    "output_rows": int(len(df)),

    "source_columns": int(pd.read_csv(SOURCE, nrows=0).shape[1]),
    "output_columns": int(df.shape[1]),

    "candidate_count": int(len(cand)),

    "reason_breakdown": {
        "multi_source_ip": multi_source,
        "multi_destination_ip": multi_destination
    },

    "reconstruction_policy": {
        "multi_source_ip":
            "HOLD / AMBIGUOUS — no automatic split",
        "multi_destination_ip":
            "HOLD / REVIEW_REQUIRED — no automatic merge",
        "normal_sessions":
            "UNCHANGED"
    },

    "safety_checks": {
        "invalid_timestamps": invalid_ts,
        "row_count_preserved": True,
        "raw_session_ids_preserved": True,
        "no_sessions_split": True,
        "no_sessions_merged": True,
        "source_modified": False,
        "attack_type_used_as_target": False,
        "ml_features_created": False
    },

    "status": "PASS"
}

with open(AUDIT, "w") as f:
    json.dump(audit, f, indent=2)

# ------------------------------------------------------------
# 12) HASH
# ------------------------------------------------------------

reconstructed_hash = sha256_file(RECONSTRUCTED)
audit_hash = sha256_file(AUDIT)

with open(HASH_FILE, "w") as f:
    f.write(
        f"{reconstructed_hash}  {RECONSTRUCTED.name}\n"
    )
    f.write(
        f"{audit_hash}  {AUDIT.name}\n"
    )

# ------------------------------------------------------------
# 13) FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2E — RECONSTRUCTION COMPLETE")
print("=" * 70)

print(f"Source rows:              {source_rows:,}")
print(f"Reconstructed rows:       {len(df):,}")
print(f"Candidates:               {len(cand):,}")
print(f"multi_source_ip:          {multi_source}")
print(f"multi_destination_ip:     {multi_destination}")

print("\nSafety:")
print("  No automatic split:     PASS")
print("  No automatic merge:     PASS")
print("  Raw IDs preserved:      PASS")
print("  Row count preserved:    PASS")
print("  Source modified:        NO")
print("  ML features created:    NO")

print("\nFiles:")
print(f"Dataset: {RECONSTRUCTED}")
print(f"Audit:   {AUDIT}")
print(f"Hashes:  {HASH_FILE}")

print("\nSHA-256:")
print(f"Dataset: {reconstructed_hash}")
print(f"Audit:   {audit_hash}")

print("\nSTATUS: PASS")

/tmp/ipykernel_906/3428377622.py:45: DtypeWarning: Columns (19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(SOURCE)


SOURCE: (58895, 39)
CANDIDATES: (47, 16)


/tmp/ipykernel_906/3428377622.py:176: DtypeWarning: Columns (19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(SOURCE)



STEP 2E — RECONSTRUCTION COMPLETE
Source rows:              58,895
Reconstructed rows:       58,895
Candidates:               47
multi_source_ip:          31
multi_destination_ip:     16

Safety:
  No automatic split:     PASS
  No automatic merge:     PASS
  Raw IDs preserved:      PASS
  Row count preserved:    PASS
  Source modified:        NO
  ML features created:    NO

Files:
Dataset: /content/session_reconstruction_v1/session_reconstructed_v1.csv
Audit:   /content/session_reconstruction_v1/session_reconstructed_v1_audit.json
Hashes:  /content/session_reconstruction_v1/session_reconstructed_v1.sha256

SHA-256:
Dataset: 81afe1c709a02d3530990f2bb9695010d6ba1b17106cc0c22600b0454b929d4e
Audit:   6baffc4c8202bc256eb494fe586ca4e179cf023541131aa9d09c3f5467acbe87

STATUS: PASS


In [ ]:

# ============================================================
# STEP 2E — BACKUP + HASH VERIFICATION CHECKPOINT
# ============================================================

from pathlib import Path
import hashlib
import json
import shutil
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1) PATHS
# ------------------------------------------------------------

SOURCE_DIR = Path("/content/session_reconstruction_v1")

BACKUP_DIR = Path("/content/checkpoints/session_reconstruction_v1_backup")

FILES_TO_BACKUP = [
    "session_reconstructed_v1.csv",
    "session_reconstructed_v1_audit.json",
    "session_reconstructed_v1.sha256",
]

# ------------------------------------------------------------
# 2) SHA-256 FUNCTION
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# 3) CHECK SOURCE FILES EXIST
# ------------------------------------------------------------

print("=" * 70)
print("STEP 2E — BACKUP + HASH VERIFICATION")
print("=" * 70)

missing = []

for filename in FILES_TO_BACKUP:
    path = SOURCE_DIR / filename

    if not path.exists():
        missing.append(str(path))

if missing:
    print("\nERROR: Required files are missing:")

    for p in missing:
        print("  ", p)

    raise FileNotFoundError(
        "Backup stopped because one or more required Step 2E files are missing."
    )

print("\n[1] Required files: PASS")


# ------------------------------------------------------------
# 4) CREATE BACKUP DIRECTORY
# ------------------------------------------------------------

BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("[2] Backup directory:")
print("   ", BACKUP_DIR)


# ------------------------------------------------------------
# 5) COPY FILES
# ------------------------------------------------------------

backup_records = []

for filename in FILES_TO_BACKUP:

    src = SOURCE_DIR / filename
    dst = BACKUP_DIR / filename

    shutil.copy2(src, dst)

    source_hash = sha256_file(src)
    backup_hash = sha256_file(dst)

    same_hash = source_hash == backup_hash

    if not same_hash:
        raise RuntimeError(
            f"HASH MISMATCH after backup: {filename}"
        )

    backup_records.append({
        "filename": filename,
        "source_path": str(src),
        "backup_path": str(dst),
        "size_bytes": src.stat().st_size,
        "sha256": source_hash,
        "source_backup_hash_match": True,
    })

    print(f"\n[BACKUP] {filename}")
    print(f"  Size: {src.stat().st_size:,} bytes")
    print(f"  SHA-256: {source_hash}")
    print("  Verification: PASS")


# ------------------------------------------------------------
# 6) VERIFY THE ORIGINAL .sha256 MANIFEST
# ------------------------------------------------------------

HASH_FILE = SOURCE_DIR / "session_reconstructed_v1.sha256"

hash_text = HASH_FILE.read_text(encoding="utf-8", errors="replace")

print("\n[3] Verifying SHA-256 manifest...")

manifest_results = []

for record in backup_records:

    filename = record["filename"]
    calculated_hash = record["sha256"]

    # Find lines containing the filename
    matching_lines = [
        line.strip()
        for line in hash_text.splitlines()
        if filename in line
    ]

    if not matching_lines:
        # The .sha256 file may contain only the hash.
        # We do not fail here; we report it.
        manifest_results.append({
            "filename": filename,
            "calculated_sha256": calculated_hash,
            "manifest_entry_found": False,
            "manifest_match": None,
        })
        continue

    manifest_hash = matching_lines[0].split()[0].lower()

    match = manifest_hash == calculated_hash.lower()

    manifest_results.append({
        "filename": filename,
        "calculated_sha256": calculated_hash,
        "manifest_hash": manifest_hash,
        "manifest_entry_found": True,
        "manifest_match": match,
    })

    if not match:
        raise RuntimeError(
            f"SHA-256 MANIFEST MISMATCH: {filename}\n"
            f"Calculated: {calculated_hash}\n"
            f"Manifest:   {manifest_hash}"
        )

    print(f"  {filename}: PASS")


# ------------------------------------------------------------
# 7) CREATE BACKUP MANIFEST
# ------------------------------------------------------------

manifest = {
    "checkpoint": "STEP 2E",
    "checkpoint_name": "session_reconstruction_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "source_directory": str(SOURCE_DIR),
    "backup_directory": str(BACKUP_DIR),

    "source_modified": False,

    "safety": {
        "raw_ids_preserved": True,
        "row_count_preserved": True,
        "automatic_split": False,
        "automatic_merge": False,
        "session_features_created": False,
        "ml_features_created": False,
    },

    "files": backup_records,

    "hash_manifest_verification": manifest_results,

    "status": "PASS",
}


MANIFEST_PATH = BACKUP_DIR / "backup_manifest.json"

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------
# 8) HASH THE BACKUP MANIFEST ITSELF
# ------------------------------------------------------------

manifest_hash = sha256_file(MANIFEST_PATH)

MANIFEST_HASH_PATH = BACKUP_DIR / "backup_manifest.sha256"

with open(MANIFEST_HASH_PATH, "w", encoding="utf-8") as f:
    f.write(f"{manifest_hash}  backup_manifest.json\n")


# ------------------------------------------------------------
# 9) FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2E CHECKPOINT COMPLETE")
print("=" * 70)

print("\nSTATUS: PASS")

print("\nBackup:")
print("  ", BACKUP_DIR)

print("\nFiles backed up:")
for record in backup_records:
    print("  PASS:", record["filename"])

print("\nBackup manifest:")
print("  ", MANIFEST_PATH)

print("\nBackup manifest SHA-256:")
print("  ", manifest_hash)

print("\nSafety:")
print("  Source modified:          NO")
print("  Raw IDs preserved:        YES")
print("  Row count preserved:      YES")
print("  Automatic split:          NO")
print("  Automatic merge:          NO")
print("  Session features:         NO")
print("  ML features:              NO")

print("\n" + "=" * 70)
print("SAFE CHECKPOINT CREATED")
print("You can continue to the next step.")
print("=" * 70)



STEP 2E — BACKUP + HASH VERIFICATION

[1] Required files: PASS
[2] Backup directory:
    /content/checkpoints/session_reconstruction_v1_backup

[BACKUP] session_reconstructed_v1.csv
  Size: 21,221,741 bytes
  SHA-256: 81afe1c709a02d3530990f2bb9695010d6ba1b17106cc0c22600b0454b929d4e
  Verification: PASS

[BACKUP] session_reconstructed_v1_audit.json
  Size: 1,075 bytes
  SHA-256: 6baffc4c8202bc256eb494fe586ca4e179cf023541131aa9d09c3f5467acbe87
  Verification: PASS

[BACKUP] session_reconstructed_v1.sha256
  Size: 197 bytes
  SHA-256: e6b226c1f39fc4815c3ad8e78afb861f6ef887fd5d2d3dae035174968bb8159b
  Verification: PASS

[3] Verifying SHA-256 manifest...
  session_reconstructed_v1.csv: PASS
  session_reconstructed_v1_audit.json: PASS

STEP 2E CHECKPOINT COMPLETE

STATUS: PASS

Backup:
   /content/checkpoints/session_reconstruction_v1_backup

Files backed up:
  PASS: session_reconstructed_v1.csv
  PASS: session_reconstructed_v1_audit.json
  PASS: session_reconstructed_v1.sha256

Backup mani

In [ ]:

# ============================================================
# DOWNLOAD STEP 2E CHECKPOINT FILES
# ============================================================

from pathlib import Path
from google.colab import files

BACKUP_DIR = Path(
    "/content/checkpoints/session_reconstruction_v1_backup"
)

FILES = [
    "session_reconstructed_v1.csv",
    "session_reconstructed_v1_audit.json",
    "session_reconstructed_v1.sha256",
    "backup_manifest.json",
    "backup_manifest.sha256",
]

print("=" * 70)
print("STEP 2E — DOWNLOAD CHECKPOINT")
print("=" * 70)

for filename in FILES:
    path = BACKUP_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    print(f"PASS: {filename} ({path.stat().st_size:,} bytes)")

print("\nAll checkpoint files found.")
print("Starting downloads...\n")

for filename in FILES:
    files.download(str(BACKUP_DIR / filename))

print("\n" + "=" * 70)
print("DOWNLOADS REQUESTED")
print("=" * 70)
print("Keep all 5 files together.")
print("Do NOT rename them.")
print("Do NOT edit the CSV.")
print("=" * 70)



STEP 2E — DOWNLOAD CHECKPOINT
PASS: session_reconstructed_v1.csv (21,221,741 bytes)
PASS: session_reconstructed_v1_audit.json (1,075 bytes)
PASS: session_reconstructed_v1.sha256 (197 bytes)
PASS: backup_manifest.json (2,705 bytes)
PASS: backup_manifest.sha256 (87 bytes)

All checkpoint files found.
Starting downloads...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DOWNLOADS REQUESTED
Keep all 5 files together.
Do NOT rename them.
Do NOT edit the CSV.
